# 🎙️ PHILIA — Audio Emotion Fine-Tuning (Optuna)
**Model:** `facebook/wav2vec2-base` → fine-tuned on MELD audio  
**Classes:** angry · disgust · fear · happy · neutral · sad · surprise  
**Optuna:** 8-trial hyperparameter search → final training with best params  
**Output:** saved to Google Drive → download and drop into PHILIA

**Runtime:** Set to `GPU` → Runtime > Change runtime type > T4 GPU


## Key Improvements Over Baseline
- **Focal Loss** replaces plain weighted cross-entropy → focuses on hard/minority classes
- **8 Optuna trials** (up from 5) → better hyperparameter coverage
- **Gradient checkpointing** → allows training with larger effective batch sizes
- **Label smoothing (0.1)** → prevents overconfidence on noisy MELD labels
- **Macro F1 tracking** → monitors per-class performance, not just overall accuracy

In [1]:
# ── Cell 1: Check GPU ──────────────────────────────────────────────────────
import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None')

CUDA available: True
GPU: NVIDIA GeForce RTX 3060 Laptop GPU


In [ ]:
# ── Cell 2: Install dependencies ───────────────────────────────────────────
!pip install -q transformers datasets accelerate evaluate soundfile librosa optuna scikit-learn

In [ ]:
# ── Cell 3: Mount Google Drive ─────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
SAVE_DIR = '/content/drive/MyDrive/PHILIA/models/audio_emotion'
os.makedirs(SAVE_DIR, exist_ok=True)
print('Model will be saved to:', SAVE_DIR)

In [ ]:
# ── Cell 4: Download MELD dataset ──────────────────────────────────────────
import os

MELD_DIR = '/content/MELD'
os.makedirs(MELD_DIR, exist_ok=True)

if not os.path.exists('/content/MELD.Raw.tar.gz'):
    print('Downloading MELD raw data...')
    !wget -q --show-progress http://web.eecs.umich.edu/~mihalcea/downloads/MELD.Raw.tar.gz -O /content/MELD.Raw.tar.gz
    print('Extracting main archive...')
    !tar -xzf /content/MELD.Raw.tar.gz -C /content/MELD --strip-components=1
    print('Done.')
else:
    print('MELD already downloaded.')

!ls /content/MELD/

In [ ]:
# ── Cell 5: Extract MELD sub-archives (train / dev / test mp4 clips) ───────
# The main archive contains sub-archives that must be extracted separately.
import tarfile, os

MELD_DIR = '/content/MELD'

for split in ['train', 'dev', 'test']:
    archive = os.path.join(MELD_DIR, f'{split}.tar.gz')
    if os.path.exists(archive):
        print(f'Extracting {split}.tar.gz...')
        with tarfile.open(archive, 'r:gz') as tar:
            tar.extractall(path=MELD_DIR, filter='data')
    else:
        print(f'WARNING: {archive} not found — skipping')

print('Done extracting all MELD sub-archives.')
!ls /content/MELD/ | head -n 10

In [ ]:
# ── Cell 6: Extract audio from MP4 clips (16 kHz mono WAV) ─────────────────
import subprocess, glob, os
from tqdm.auto import tqdm

AUDIO_DIR = '/content/MELD_audio'
os.makedirs(AUDIO_DIR, exist_ok=True)

mp4_files = glob.glob('/content/MELD/**/*.mp4', recursive=True)
print(f'Found {len(mp4_files)} mp4 files')

for mp4 in tqdm(mp4_files, desc='Extracting audio'):
    wav_out = os.path.join(AUDIO_DIR, os.path.basename(mp4).replace('.mp4', '.wav'))
    if not os.path.exists(wav_out):
        subprocess.run(
            ['ffmpeg', '-i', mp4, '-ar', '16000', '-ac', '1', '-loglevel', 'error', wav_out],
            check=False
        )

print(f'Audio files extracted: {len(glob.glob(AUDIO_DIR + "/*.wav"))}')

In [ ]:
# ── Cell 7: Load CSV labels and build DataFrames ───────────────────────────
import pandas as pd
import os

!wget -q https://raw.githubusercontent.com/declare-lab/MELD/master/data/MELD/train_sent_emo.csv -O /content/train.csv
!wget -q https://raw.githubusercontent.com/declare-lab/MELD/master/data/MELD/dev_sent_emo.csv   -O /content/dev.csv
!wget -q https://raw.githubusercontent.com/declare-lab/MELD/master/data/MELD/test_sent_emo.csv  -O /content/test.csv

EMOTION_MAP = {
    'anger':    'angry',
    'disgust':  'disgust',
    'fear':     'fear',
    'joy':      'happy',
    'neutral':  'neutral',
    'sadness':  'sad',
    'surprise': 'surprise',
}
LABELS   = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']
LABEL2ID = {l: i for i, l in enumerate(LABELS)}
ID2LABEL = {i: l for i, l in enumerate(LABELS)}

AUDIO_DIR = '/content/MELD_audio'

def load_split(csv_path, split_name):
    df = pd.read_csv(csv_path)
    df['wav_path'] = df.apply(
        lambda r: os.path.join(
            AUDIO_DIR,
            f'dia{r["Dialogue_ID"]}_utt{r["Utterance_ID"]}.wav'
        ), axis=1
    )
    df['canonical'] = df['Emotion'].str.lower().map(EMOTION_MAP)
    df = df[df['canonical'].notna()]
    df = df[df['wav_path'].apply(os.path.exists)]
    df['label'] = df['canonical'].map(LABEL2ID)
    print(f'{split_name}: {len(df)} samples | dist: {dict(df["canonical"].value_counts())}')
    return df[['wav_path', 'label', 'canonical']].reset_index(drop=True)

train_df = load_split('/content/train.csv', 'TRAIN')
val_df   = load_split('/content/dev.csv',   'VAL')
test_df  = load_split('/content/test.csv',  'TEST')

In [ ]:
# ── Cell 8: HuggingFace Dataset + feature extraction ───────────────────────
import torch, librosa, numpy as np
from datasets import Dataset
from transformers import Wav2Vec2FeatureExtractor

MODEL_CHECKPOINT = 'facebook/wav2vec2-base'
SAMPLE_RATE      = 16000
MAX_DURATION     = 8.0   # 8s to capture more emotional context

feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(MODEL_CHECKPOINT)

def df_to_dataset(df):
    return Dataset.from_pandas(df[['wav_path', 'label']])

def preprocess(batch):
    arrays = []
    for path in batch['wav_path']:
        arr, _ = librosa.load(path, sr=SAMPLE_RATE, mono=True, duration=MAX_DURATION)
        arrays.append(arr)
    inputs = feature_extractor(
        arrays,
        sampling_rate=SAMPLE_RATE,
        padding=True,
        truncation=True,
        max_length=int(SAMPLE_RATE * MAX_DURATION),
        return_tensors='np',
    )
    return {'input_values': inputs.input_values, 'labels': batch['label']}

print('Preprocessing train...')
train_ds = df_to_dataset(train_df).map(preprocess, batched=True, batch_size=32, remove_columns=['wav_path'])
print('Preprocessing val...')
val_ds   = df_to_dataset(val_df).map(preprocess,   batched=True, batch_size=32, remove_columns=['wav_path'])
print('Preprocessing test...')
test_ds  = df_to_dataset(test_df).map(preprocess,  batched=True, batch_size=32, remove_columns=['wav_path'])

train_ds.set_format('torch')
val_ds.set_format('torch')
test_ds.set_format('torch')
print('Datasets ready.')

In [ ]:
# ── Cell 9: Metrics · DataCollator · Class Weights ─────────────────────────
import evaluate, torch.nn as nn
from sklearn.metrics import f1_score
from collections import Counter
from transformers import Trainer, TrainingArguments, EarlyStoppingCallback
from dataclasses import dataclass
from typing import Union

accuracy_metric = evaluate.load('accuracy')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_metric.compute(predictions=preds, references=labels)['accuracy']
    f1      = f1_score(labels, preds, average='weighted')
    f1_mac  = f1_score(labels, preds, average='macro')   
    return {'accuracy': acc, 'f1': f1, 'f1_macro': f1_mac}

@dataclass
class DataCollatorWithPadding:
    feature_extractor: Wav2Vec2FeatureExtractor
    padding: Union[bool, str] = True

    def __call__(self, features):
        input_values = [{'input_values': f['input_values']} for f in features]
        labels = torch.tensor([f['labels'] for f in features], dtype=torch.long)
        batch = self.feature_extractor.pad(input_values, padding=self.padding, return_tensors='pt')
        batch['labels'] = labels
        return batch

data_collator = DataCollatorWithPadding(feature_extractor=feature_extractor)

# Class weights — neutral ≈ 47% of MELD train; boost minority classes
label_counts  = Counter(train_df['label'].tolist())
total         = sum(label_counts.values())
class_weights = torch.tensor(
    [total / (len(LABELS) * label_counts.get(i, 1)) for i in range(len(LABELS))],
    dtype=torch.float,
)
print('Class weights:', {ID2LABEL[i]: f'{w:.3f}' for i, w in enumerate(class_weights)})

In [ ]:
# ── Cell 10: model_init · FocalLossTrainer · hp_space ─────────────────────
# Focal Loss vs plain weighted CE
import optuna
from transformers import Wav2Vec2ForSequenceClassification

optuna.logging.set_verbosity(optuna.logging.WARNING)

FOCAL_GAMMA = 2.0   

def focal_loss(logits, labels, gamma=FOCAL_GAMMA):
    """Focal loss with class weighting."""
    weights = class_weights.to(logits.device)
    ce_loss = nn.functional.cross_entropy(logits, labels, weight=weights, reduction='none')
    pt = torch.exp(-ce_loss)       
    return ((1 - pt) ** gamma * ce_loss).mean()

def model_init(trial=None):
    """Fresh model for every Optuna trial — required by hyperparameter_search."""
    m = Wav2Vec2ForSequenceClassification.from_pretrained(
        MODEL_CHECKPOINT,
        num_labels=len(LABELS),
        label2id=LABEL2ID,
        id2label=ID2LABEL,
        ignore_mismatched_sizes=True,
    )
    m.freeze_feature_encoder()   
    return m

class FocalLossTrainer(Trainer):
    """Trainer with Focal Loss to handle class imbalance better than plain CE."""
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        loss = focal_loss(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

def hp_space(trial):
    """Search space for Optuna — feel free to expand ranges if you have time."""
    return {
        'learning_rate':               trial.suggest_float('learning_rate', 3e-6, 5e-4, log=True),
        'per_device_train_batch_size': trial.suggest_categorical('per_device_train_batch_size', [8, 16]),
        'num_train_epochs':            trial.suggest_int('num_train_epochs', 4, 10),
        'weight_decay':                trial.suggest_float('weight_decay', 1e-4, 0.15, log=True),
        'warmup_ratio':                trial.suggest_float('warmup_ratio', 0.03, 0.25),
    }

print('model_init, FocalLossTrainer and hp_space defined.')

---
## Phase 1 — Optuna Hyperparameter Search
Runs **8 short trials** to discover the best learning rate, batch size, epochs, weight decay, and warmup ratio.
Each trial trains a fresh model for a few epochs. Optuna prunes poor trials early to save time.

This does NOT produce your final model. It only finds the best settings for Phase 2.

In [ ]:
# ── Cell 11: Optuna Hyperparameter Search (8 trials) ──────────────────────
# PHASE 1 — finds best hyperparameters

search_args = TrainingArguments(
    output_dir='/content/optuna_search',
    eval_strategy='epoch',
    save_strategy='no',          
    logging_steps=100,
    fp16=True,
    gradient_accumulation_steps=2,
    gradient_checkpointing=True, 
    report_to='none',
)

search_trainer = FocalLossTrainer(
    model_init=model_init,       
    args=search_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    data_collator=data_collator,
)

print('Running Optuna search — 8 trials × up to 10 epochs each...')
print('(Optuna will prune bad trials early — total time ~30-60 min on T4)')
best_run = search_trainer.hyperparameter_search(
    direction='maximize',
    backend='optuna',
    hp_space=hp_space,
    n_trials=8,
    compute_objective=lambda m: m['eval_f1'],   
)

print(f'\n Phase 1 complete!')
print(f'   Best trial  : {best_run.run_id}')
print(f'   eval_f1     : {best_run.objective:.4f}')
print('   Best hyperparameters:')
for k, v in best_run.hyperparameters.items():
    print(f'     {k}: {v}')

---
## Phase 2 — Final Training with Best Hyperparameters
Now that we know the best settings, we train a **fresh model all the way through** using those settings.
This is where the real production model is produced and saved.  

In [ ]:
# ── Cell 12: Final Training (PHASE 2) ─────────────────────────────────────

hp = best_run.hyperparameters

final_args = TrainingArguments(
    output_dir='/content/wav2vec2_audio_emotion',
    num_train_epochs=hp.get('num_train_epochs', 6),
    per_device_train_batch_size=hp.get('per_device_train_batch_size', 8),
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    gradient_checkpointing=True,
    warmup_ratio=hp.get('warmup_ratio', 0.1),
    learning_rate=hp.get('learning_rate', 1e-5),
    weight_decay=hp.get('weight_decay', 0.01),
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    greater_is_better=True,
    logging_steps=50,
    fp16=True,
    report_to='none',
    label_smoothing_factor=0.1,  
)

final_trainer = FocalLossTrainer(
    model=model_init(),              
    args=final_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    data_collator=data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

print(' Starting Phase 2 — final training with best hyperparameters...')
final_trainer.train()

In [ ]:
# ── Cell 13: Evaluate on test set ─────────────────────────────────────────
from sklearn.metrics import classification_report

test_results = final_trainer.evaluate(test_ds)
print('Test results:', test_results)

pred_output = final_trainer.predict(test_ds)
preds = np.argmax(pred_output.predictions, axis=-1)
print(classification_report(
    pred_output.label_ids, preds,
    target_names=LABELS,
    digits=3
))

In [ ]:
# ── Cell 14: Save to Google Drive ─────────────────────────────────────────
print(f'Saving model to {SAVE_DIR} ...')
final_trainer.model.save_pretrained(SAVE_DIR)
feature_extractor.save_pretrained(SAVE_DIR)
print('Done! Files in Drive:')
import os
for f in os.listdir(SAVE_DIR):
    size = os.path.getsize(os.path.join(SAVE_DIR, f)) / 1e6
    print(f'  {f}  ({size:.1f} MB)')

In [ ]:
# ── Cell 15: (Optional) Push to Hugging Face Hub ──────────────────────────
# from huggingface_hub import login
# login(token='YOUR_HF_TOKEN')
# final_trainer.model.push_to_hub('YOUR_HF_USERNAME/philia-audio-emotion')
# feature_extractor.push_to_hub('YOUR_HF_USERNAME/philia-audio-emotion')
print('Skipped (optional). Uncomment above lines to push to HF Hub.')